# FGSM 공격 이미지 방어 실험

### 사전 준비
Google Drive에 아래 파일 올려두기:
```
내 드라이브/hanium-aml-defense/hanium_attack_outputs.zip
내 드라이브/hanium-aml-defense/best.pt
```

In [ ]:
# 1. Drive 마운트
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. 레포 클론
%cd /content
!rm -rf hanium-aml-defense
!git clone https://github.com/Yaho03/26_HC160.git hanium-aml-defense
%cd /content/hanium-aml-defense

In [ ]:
# 3. 공격 결과 + 체크포인트 복원
import zipfile, shutil
from pathlib import Path

DRIVE = Path('/content/drive/MyDrive/hanium-aml-defense')
REPO = Path('/content/hanium-aml-defense')

# 공격 결과 압축 해제
zip_path = DRIVE / 'hanium_attack_outputs.zip'
assert zip_path.exists(), f'ZIP 없음: {zip_path}'
with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall(REPO)
print('공격 결과 복원 완료')

# 체크포인트 복사
ckpt_dir = REPO / 'checkpoints' / 'face_resnet50_lfw10'
ckpt_dir.mkdir(parents=True, exist_ok=True)
shutil.copy(DRIVE / 'best.pt', ckpt_dir / 'best.pt')
print('체크포인트 복원 완료')

# 확인
import pandas as pd
idx = pd.read_csv(REPO / 'outputs/attacks/attack_index.csv')
fgsm = idx[idx['attack_family'] == 'fgsm']
print(f'\nFGSM 공격 샘플: {len(fgsm)}장')
print(f'그중 공격 성공: {fgsm["success_on_clean"].sum()}장')

In [ ]:
# 4. FGSM 방어 실행 (먼저 10장으로 테스트)
%cd /content/hanium-aml-defense/src
!python defense_fgsm_only.py --limit 10

In [ ]:
# 5. 전체 FGSM 샘플 방어 실행
!python defense_fgsm_only.py

In [ ]:
# 6. 결과 확인
import pandas as pd

df = pd.read_csv('../outputs/defenses/fgsm_defense_results.csv')

print('=== 방어 기법별 결과 ===')
for defense, g in df.groupby('defense'):
    n = len(g)
    n_def = (g['attack_success_after_defense'] == False).sum()
    n_rec = g['recovered'].sum()
    conf_drop = (g['target_conf_before'] - g['target_conf_after']).mean()
    print(f'\n{defense}:')
    print(f'  Defense Success Rate: {n_def}/{n} ({100*n_def/n:.1f}%)')
    print(f'  Recovery Rate:        {n_rec}/{n} ({100*n_rec/n:.1f}%)')
    print(f'  Avg Confidence Drop:  {conf_drop:.4f}')

In [ ]:
# 7. 방어 전후 시각화 (샘플 5장)
import torch
import torchvision.models as models
import torchvision.transforms as T
import torch.nn as nn
import matplotlib.pyplot as plt
from PIL import Image
import io
import numpy as np

REPO = Path('/content/hanium-aml-defense')
ckpt = torch.load(REPO / 'checkpoints/face_resnet50_lfw10/best.pt',
                   map_location='cpu', weights_only=False)
model = models.resnet50(weights=None)
model.fc = nn.Linear(2048, 10)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
classes = ckpt['classes']

transform = T.Compose([T.Resize((224, 224)), T.ToTensor()])

def jpeg_defense(img_t, q=75):
    img = T.ToPILImage()(img_t)
    buf = io.BytesIO()
    img.save(buf, format='JPEG', quality=q)
    buf.seek(0)
    return T.ToTensor()(Image.open(buf).convert('RGB').resize((224, 224)))

# FGSM 공격 성공 샘플 5장
idx = pd.read_csv(REPO / 'outputs/attacks/attack_index.csv')
samples = idx[(idx['attack_family']=='fgsm') &
              (idx['success_on_clean']==True)].head(5)

fig, axes = plt.subplots(5, 2, figsize=(8, 20))
for i, (_, row) in enumerate(samples.iterrows()):
    adv_img = transform(Image.open(REPO / row['adv_file']).convert('RGB'))
    defended = jpeg_defense(adv_img, 75)

    pred_before = model(adv_img.unsqueeze(0)).argmax(1).item()
    pred_after = model(defended.unsqueeze(0)).argmax(1).item()
    true_name = row['true_name']

    axes[i][0].imshow(adv_img.permute(1,2,0).numpy())
    axes[i][0].set_title(f'공격됨: {classes[pred_before]}\n(정답: {true_name})', color='red')
    axes[i][0].axis('off')

    color = 'green' if pred_after == int(row['true_label']) else 'red'
    axes[i][1].imshow(defended.permute(1,2,0).numpy())
    axes[i][1].set_title(f'방어후: {classes[pred_after]}\n(정답: {true_name})', color=color)
    axes[i][1].axis('off')

plt.suptitle('FGSM 공격 이미지 vs JPEG 방어 후', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# 8. Drive에 결과 백업
dst = Path('/content/drive/MyDrive/hanium-aml-defense/results')
dst.mkdir(parents=True, exist_ok=True)
!cp /content/hanium-aml-defense/outputs/defenses/fgsm_defense_results.csv {dst}/
print(f'백업 완료: {dst}')